# E2E Flow Test: Qdrant Hybrid Search (Dense + Sparse + RRF)
This notebook tests the E2E flow for insertion and retrieval from Qdrant using dense and sparse vectors, along with Reciprocal Rank Fusion (RRF).

In [1]:
import logging
from arxiv_scholar.schema import Chunk
from arxiv_scholar.embedding.fastembed_embedder import FastEmbedEmbedder, SparseBM25Embedder
from arxiv_scholar.storage.qdrant_store import QdrantVectorStore

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

print("Initializing embedders...")
dense_embedder = FastEmbedEmbedder(
    model_name="BAAI/bge-small-en-v1.5",
    batch_size=2
)
sparse_embedder = SparseBM25Embedder(batch_size=2)

print("Connecting to Qdrant...")
store = QdrantVectorStore(
    collection_name="test_hybrid_search_nb",
    host="localhost",
    port=6333
)

print("Ensuring collection exists...")
store.ensure_collection(dimension=dense_embedder.dimension)

print("Creating mock chunks...")
texts = [
    "Machine learning is a field of study in artificial intelligence.",
    "Quantum computing is a rapidly-emerging technology that harnesses the laws of quantum mechanics.",
    "Deep learning is a subset of machine learning, which is essentially a neural network with three or more layers."
]

import hashlib
chunks = []
for i, text in enumerate(texts):
    chunk_id = hashlib.sha256(text.encode()).hexdigest()
    chunks.append(
        Chunk(
            id=chunk_id,
            document_id="doc_1",
            content=text,
            metadata={"index": i}
        )
    )

print("Embedding dense vectors...")
dense_vectors = dense_embedder.embed(texts)

print("Embedding sparse vectors...")
sparse_vectors = sparse_embedder.embed(texts)

print("Upserting to Qdrant...")
store.upsert(chunks, dense_vectors, sparse_vectors)
print("Upsert successful!")

# Test retrieval
query_text = "What is deep learning?"
print(f"\nSearching for: '{query_text}'")
query_dense = dense_embedder.embed([query_text])[0]
query_sparse = sparse_embedder.embed([query_text])[0]

print("Performing hybrid search...")
results = store.hybrid_search(
    query_vector=query_dense,
    sparse_vector=query_sparse,
    top_k=2
)

print("\nResults:")
for i, res in enumerate(results):
    print(f"Rank {i+1}: Score={res['score']} | Content: {res['content']}")

/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-26 11:09:36,197 [INFO] Loading FastEmbed model 'BAAI/bge-small-en-v1.5' (CPU/ONNX)...


Initializing embedders...


2026-05-26 11:09:36,397 [INFO] Model loaded. Dimension: 384, Device: CPU (ONNX)
2026-05-26 11:09:36,398 [INFO] Loading FastEmbed sparse model 'Qdrant/bm25'...
2026-05-26 11:09:36,566 [INFO] QdrantVectorStore connected to localhost:6333, collection=test_hybrid_search_nb
2026-05-26 11:09:36,569 [INFO] HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"
2026-05-26 11:09:36,609 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


Connecting to Qdrant...
Ensuring collection exists...


/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
2026-05-26 11:09:37,133 [INFO] HTTP Request: PUT http://localhost:6333/collections/test_hybrid_search_nb "HTTP/1.1 200 OK"
2026-05-26 11:09:37,135 [INFO] Created collection 'test_hybrid_search_nb' (dim=384, distance=COSINE).
2026-05-26 11:09:37,163 [INFO] HTTP Request: PUT http://localhost:6333/collections/test_hybrid_search_nb/points?wait=true "HTTP/1.1 200 OK"
2026-05-26 11:09:37,176 [INFO] HTTP Request: POST http://localhost:6333/collections/test_hybrid_search_nb/points/query "HTTP/1.1 200 OK"


Creating mock chunks...
Embedding dense vectors...
Embedding sparse vectors...
Upserting to Qdrant...
Upsert successful!

Searching for: 'What is deep learning?'
Performing hybrid search...

Results:
Rank 1: Score=1.0 | Content: Deep learning is a subset of machine learning, which is essentially a neural network with three or more layers.
Rank 2: Score=0.6666667 | Content: Machine learning is a field of study in artificial intelligence.
